In [98]:
import pandas as pd
import numpy as np

In [123]:
df_bureau = pd.read_csv(r"C:\Users\Lenovo\Desktop\bureau.csv")

In [100]:
df_bureau.head(2)

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN


In [124]:
cols_to_fill_zero = ['AMT_CREDIT_SUM_DEBT','AMT_CREDIT_SUM_LIMIT','AMT_CREDIT_SUM_OVERDUE','AMT_CREDIT_MAX_OVERDUE']

for col in cols_to_fill_zero:
    df_bureau[col]=df_bureau[col].fillna(0)

In [125]:
df_bureau.isna().sum()

SK_ID_CURR                      0
SK_ID_BUREAU                    0
CREDIT_ACTIVE                   0
CREDIT_CURRENCY                 0
DAYS_CREDIT                     0
CREDIT_DAY_OVERDUE              0
DAYS_CREDIT_ENDDATE        105553
DAYS_ENDDATE_FACT          633653
AMT_CREDIT_MAX_OVERDUE          0
CNT_CREDIT_PROLONG              0
AMT_CREDIT_SUM                 13
AMT_CREDIT_SUM_DEBT             0
AMT_CREDIT_SUM_LIMIT            0
AMT_CREDIT_SUM_OVERDUE          0
CREDIT_TYPE                     0
DAYS_CREDIT_UPDATE              0
AMT_ANNUITY               1226791
dtype: int64

In [126]:
df_bureau['is_credit_active'] = df_bureau['DAYS_ENDDATE_FACT'].isnull().astype(int)

In [127]:
df_bureau['AMT_CREDIT_SUM'] = df_bureau['AMT_CREDIT_SUM'].fillna(0)

In [129]:
df_bureau['DAYS_CREDIT_ENDDATE']=df_bureau['DAYS_CREDIT_ENDDATE'].fillna(df_bureau['DAYS_CREDIT_ENDDATE'].median())

In [130]:
df_bureau['DAYS_ENDDATE_FACT']=df_bureau['DAYS_ENDDATE_FACT'].fillna(0)

In [131]:
df_bureau.drop(columns='DAYS_CREDIT_UPDATE',inplace=True)

In [132]:
df_bureau['AMT_CREDIT_SUM']=df_bureau['AMT_CREDIT_SUM'].fillna(0)

In [133]:
df_bureau['AMT_CREDIT_SUM_DEBT']=df_bureau['AMT_CREDIT_SUM_DEBT'].fillna(0)

In [134]:
df_bureau['AMT_CREDIT_SUM_LIMIT']=df_bureau['AMT_CREDIT_SUM_LIMIT'].fillna(0)

In [135]:
df_bureau['AMT_CREDIT_MAX_OVERDUE']=df_bureau['AMT_CREDIT_MAX_OVERDUE'].fillna(0)

In [136]:
df_bureau.drop(columns=['CREDIT_CURRENCY'],inplace=True)

In [137]:
df_bureau['CREDIT_TYPE'].value_counts()

CREDIT_TYPE
Consumer credit                                 1251615
Credit card                                      402195
Car loan                                          27690
Mortgage                                          18391
Microloan                                         12413
Loan for business development                      1975
Another type of loan                               1017
Unknown type of loan                                555
Loan for working capital replenishment              469
Cash loan (non-earmarked)                            56
Real estate loan                                     27
Loan for the purchase of equipment                   19
Loan for purchase of shares (margin lending)          4
Mobile operator loan                                  1
Interbank credit                                      1
Name: count, dtype: int64

In [138]:
top_types = ['Consumer credit', 'Credit card', 'Car loan', 'Mortgage', 
             'Microloan', 'Loan for business development']

df_bureau['CREDIT_TYPE'] = df_bureau['CREDIT_TYPE'].apply(lambda x: x if x in top_types else 'Other_Loan_Types')

credit_type_dummies = pd.get_dummies(df_bureau['CREDIT_TYPE'], prefix='type',dtype=int)
df_bureau = pd.concat([df_bureau, credit_type_dummies], axis=1)
df_bureau.drop('CREDIT_TYPE', axis=1, inplace=True)

In [139]:
df_bureau.drop(columns='CREDIT_ACTIVE',inplace=True)

In [140]:
df_bureau.columns = [col.lower().strip() for col in df_bureau.columns]
bureau_agg = df_bureau.groupby('sk_id_curr').agg({
    
    # say
    'sk_id_bureau': 'count',
    
    # overdue
    'credit_day_overdue': ['mean', 'max'],
    
    # debt
    'amt_credit_sum': 'sum',
    'amt_credit_sum_debt': 'sum',
    'amt_credit_sum_overdue': 'sum',
    
    # behavior
    'cnt_credit_prolong': 'sum',
    
    # active loans
    'is_credit_active': 'sum',
    
    # credit types
    'type_credit card': 'sum',
    'type_consumer credit': 'sum',
    'type_mortgage': 'sum'
    
})

In [142]:
print(bureau_agg.columns)

MultiIndex([(          'sk_id_bureau', 'count'),
            (    'credit_day_overdue',  'mean'),
            (    'credit_day_overdue',   'max'),
            (        'amt_credit_sum',   'sum'),
            (   'amt_credit_sum_debt',   'sum'),
            ('amt_credit_sum_overdue',   'sum'),
            (    'cnt_credit_prolong',   'sum'),
            (      'is_credit_active',   'sum'),
            (      'type_credit card',   'sum'),
            (  'type_consumer credit',   'sum'),
            (         'type_mortgage',   'sum')],
           )


In [143]:
bureau_agg['debt_ratio'] = (
    bureau_agg['amt_credit_sum_debt',   'sum'] /
    bureau_agg[ 'amt_credit_sum',   'sum']
)

bureau_agg['overdue_ratio'] = (
    bureau_agg['amt_credit_sum_overdue',   'sum'] /
    bureau_agg[ 'amt_credit_sum',   'sum']
)

In [144]:
bureau_agg

sk_id_bureau credit_day_overdue     amt_credit_sum  \
                  count               mean max            sum   
sk_id_curr                                                      
100001                7                0.0   0    1453365.000   
100002                8                0.0   0     865055.565   
100003                4                0.0   0    1017400.500   
100004                2                0.0   0     189037.800   
100005                3                0.0   0     657126.000   
...                 ...                ...  ..            ...   
456249               13                0.0   0    3693858.660   
456250                3                0.0   0    3086459.550   
456253                4                0.0   0    3960000.000   
456254                1                0.0   0      45000.000   
456255               11                0.0   0    3801919.500   

           amt_credit_sum_debt amt_credit_sum_overdue cnt_credit_prolong  \
                           sum                    sum                sum   
sk_id_curr                                                                 
100001              596686.500                    0.0                  0   
100002              245781.000                    0.0                  0   
100003                   0.000                    0.0                  0   
100004                   0.000                    0.0                  0   
100005              568408.500                    0.0                  0   
...                        ...                    ...                ...   
456249              163071.000                    0.0                  0   
456250             2232040.095                    0.0                  0   
456253             1795833.000                    0.0                  0   
456254                   0.000                    0.0                  0   
456255             1534913.010                    0.0                  1   

           is_credit_active type_credit card type_consumer credit  \
                        sum              sum                  sum   
sk_id_curr                                                          
100001                    3                0                    7   
100002                    2                4                    4   
100003                    1                2                    2   
100004                    0                0                    2   
100005                    2                1                    2   
...                     ...              ...                  ...   
456249                    1                3                    9   
456250                    2                1                    2   
456253                    2                1                    3   
456254                    0                0                    1   
456255                    5                2                    9   

           type_mortgage debt_ratio overdue_ratio  
                     sum                           
sk_id_curr                                         
100001                 0   0.410555           0.0  
100002                 0   0.284122           0.0  
100003                 0   0.000000           0.0  
100004                 0   0.000000           0.0  
100005                 0   0.864992           0.0  
...                  ...        ...           ...  
456249                 0   0.044147           0.0  
456250                 0   0.723172           0.0  
456253                 0   0.453493           0.0  
456254                 0   0.000000           0.0  
456255                 0   0.403721           0.0  

[305811 rows x 13 columns]

In [145]:
bureau_agg.to_csv(r"C:\Users\Lenovo\Desktop\Final Project\Cleaned_Datas\bureau_final", index=False)

In [146]:
import sqlite3

db_path = r'C:\Users\Lenovo\Desktop\Final Project\Database\bank_credit.db'
conn = sqlite3.connect(db_path)

bureau_agg.to_sql('bureau_final', conn, if_exists='replace', index=False)

conn.close()
print("Data uğurla DBeaver-ə göndərildi!")

Data uğurla DBeaver-ə göndərildi!


In [157]:
df_previous=pd.read_csv(r"C:\Users\Lenovo\Desktop\previous_application.csv")

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

df_previous.head(5)

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,FLAG_LAST_APPL_PER_CONTRACT,NFLAG_LAST_APPL_IN_DAY,RATE_DOWN_PAYMENT,RATE_INTEREST_PRIMARY,RATE_INTEREST_PRIVILEGED,NAME_CASH_LOAN_PURPOSE,NAME_CONTRACT_STATUS,DAYS_DECISION,NAME_PAYMENT_TYPE,CODE_REJECT_REASON,NAME_TYPE_SUITE,NAME_CLIENT_TYPE,NAME_GOODS_CATEGORY,NAME_PORTFOLIO,NAME_PRODUCT_TYPE,CHANNEL_TYPE,SELLERPLACE_AREA,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,Y,1,0.0,0.182832,0.867336,XAP,Approved,-73,Cash through the bank,XAP,NaN,Repeater,Mobile,POS,XNA,Country-wide,35,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,Y,1,NaN,NaN,NaN,XNA,Approved,-164,XNA,XAP,Unaccompanied,Repeater,XNA,Cash,x-sell,Contact center,-1,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,Y,1,NaN,NaN,NaN,XNA,Approved,-301,Cash through the bank,XAP,"Spouse, partner",Repeater,XNA,Cash,x-sell,Credit and cash offices,-1,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,Y,1,NaN,NaN,NaN,XNA,Approved,-512,Cash through the bank,XAP,NaN,Repeater,XNA,Cash,x-sell,Credit and cash offices,-1,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,Y,1,NaN,NaN,NaN,Repairs,Refused,-781,Cash through the bank,HC,NaN,Repeater,XNA,Cash,walk-in,Credit and cash offices,-1,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [158]:

df_previous.columns = [col.lower() for col in df_previous.columns]

# 2. Yeni feature-ların yaradılması
df_previous['asked_credit_ratio'] = df_previous['amt_application'] / df_previous['amt_credit'].replace(0, 1)
df_previous['has_down_payment'] = (df_previous['amt_down_payment'] > 0).astype(int)

# 3. Anomaliyaların təmizlənməsi (365243 -> NaN)
days_cols = ['days_first_drawing', 'days_first_due', 'days_last_due_1st_version', 'days_last_due', 'days_termination']
for col in days_cols:
    df_previous[col].replace(365243, np.nan, inplace=True)

# 4. Lazımsız sütunların silinməsi
cols_to_drop = [
    'weekday_appr_process_start', 'hour_appr_process_start', 
    'flag_last_appl_per_contract', 'nflag_last_appl_in_day',
    'rate_interest_primary', 'rate_interest_privileged', 'name_type_suite'
]
df_previous.drop(columns=cols_to_drop, inplace=True)


In [159]:
df_previous['amt_down_payment']=df_previous['amt_down_payment'].fillna(0)

In [160]:
# Hər bir status üçün yeni sütunlar yaradırıq (0 və 1 olaraq)
df_previous['is_approved'] = (df_previous['name_contract_status'] == 'Approved').astype(int)
df_previous['is_refused'] = (df_previous['name_contract_status'] == 'Refused').astype(int)

In [ ]:
prev_agg = df_previous.groupby('sk_id_curr').agg({

    # =====================
    # COUNT & BEHAVIOR
    # =====================
    'sk_id_prev': 'count',                 # total applications
    'is_approved': 'sum',                  # approved count
    'is_refused': 'sum',                   # refused count

    # =====================
    # CREDIT AMOUNTS (CRITICAL ADD)
    # =====================
    'amt_credit': ['mean', 'sum'],         # avg + total credit
    'amt_application': 'mean',             # avg requested amount

    # =====================
    # PAYMENT INFO
    # =====================
    'amt_annuity': ['mean', 'max'],        # monthly payments
    'cnt_payment': 'mean',                 # loan duration

    # =====================
    # RATIOS
    # =====================
    'asked_credit_ratio': 'mean',          # credit/application ratio

    # =====================
    # TIME (RECENCY)
    # =====================
    'days_decision': ['min', 'mean', 'max'],

    # =====================
    # DOWN PAYMENT
    # =====================
    'amt_down_payment': 'sum',             # total down payment
    'has_down_payment': 'sum'              # count of DP usage

}).reset_index()

In [162]:
# =========================
# AGGREGATION
# =========================
prev_agg = df_previous.groupby('sk_id_curr').agg({

    # Behavior
    'sk_id_prev': 'count',
    'is_approved': 'sum',
    'is_refused': 'sum',

    # CREDIT (ƏN VACİB ƏLAVƏ)
    'amt_credit': ['mean', 'sum'],
    'amt_application': 'mean',

    # Payments
    'amt_annuity': ['mean', 'max'],
    'cnt_payment': 'mean',

    # Ratios
    'asked_credit_ratio': 'mean',

    # Time (RECENCY)
    'days_decision': ['min', 'mean', 'max'],

    # Down payment
    'amt_down_payment': 'sum',
    'has_down_payment': 'sum'

}).reset_index()


# =========================
# COLUMN NAMES FIX
# =========================
prev_agg.columns = [
    'sk_id_curr',

    'sk_id_prev_count',
    'is_approved_sum',
    'is_refused_sum',

    'avg_credit',
    'total_credit',

    'avg_application',

    'amt_annuity_mean',
    'amt_annuity_max',

    'avg_cnt_payment',

    'asked_credit_ratio_mean',

    'days_decision_min',
    'days_decision_mean',
    'days_decision_max',

    'total_down_payment',
    'has_down_payment_sum'
]


# =========================
# FEATURE ENGINEERING
# =========================

# Approval / Refusal
prev_agg['approval_rate'] = (
    prev_agg['is_approved_sum'] / prev_agg['sk_id_prev_count']
)

prev_agg['refused_ratio'] = (
    prev_agg['is_refused_sum'] / prev_agg['sk_id_prev_count']
)

# Credit behavior
prev_agg['credit_per_application'] = (
    prev_agg['total_credit'] / prev_agg['sk_id_prev_count']
)

# BANK TRUST GAP (🔥 çox güclü feature)
prev_agg['credit_gap'] = (
    prev_agg['avg_application'] - prev_agg['avg_credit']
)

# Safety
prev_agg = prev_agg.fillna(0)

In [166]:
#insanin goturduyu yuke gore ayliq yuku
prev_agg['annuity_to_credit_ratio'] = (
    prev_agg['amt_annuity_mean'] / prev_agg['avg_credit']
)

#ilkin odenis etme ratio'su ,ilkin idenisi edenler daha az riskli ola biler
prev_agg['down_payment_ratio'] = (
    prev_agg['total_down_payment'] / prev_agg['total_credit']
)

# muraciet etme frequency'si cox olanlar daha riskli ola bilerler
prev_agg['applications_per_day'] = (
    prev_agg['sk_id_prev_count'] / (abs(prev_agg['days_decision_min']) + 1)
)

# musterinin kecmisine gore bank ucun bunun approved ve refused'lari arasindaki balansa baxir ,ona uygun olaraq bank bunun riskli olub olmadigina qerar vere biler
prev_agg['approval_minus_refusal'] = (
    prev_agg['is_approved_sum'] - prev_agg['is_refused_sum']
)

In [167]:
import sqlite3

db_path = r'C:\Users\Lenovo\Desktop\Final Project\Database\bank_credit.db'
conn = sqlite3.connect(db_path)

# 'prev_final' adı ilə bazaya yazırıq
prev_agg.to_sql('previous_final', conn, if_exists='replace', index=False)

conn.close()
print("Təbriklər! Previous Application cədvəli artıq SQL-dədir.")

Təbriklər! Previous Application cədvəli artıq SQL-dədir.


In [168]:
df_posh_cash=pd.read_csv(r"C:\Users\Lenovo\Desktop\POS_CASH_balance.csv")
df_posh_cash.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0


In [170]:
import numpy as np

pos = df_posh_cash

# =========================
# AGGREGATION
# =========================
pos_agg = pos.groupby('SK_ID_CURR').agg({

    # how many contracts
    'SK_ID_PREV': 'count',

    # installment behavior
    'CNT_INSTALMENT': ['mean', 'max'],
    'CNT_INSTALMENT_FUTURE': ['mean', 'min'],

    # delay behavior (VERY IMPORTANT)
    'SK_DPD': ['mean', 'max'],
    'SK_DPD_DEF': ['mean', 'max'],

    # recency
    'MONTHS_BALANCE': ['min', 'max']

}).reset_index()


# =========================
# FLATTEN COLUMN NAMES
# =========================
pos_agg.columns = [
    'sk_id_curr',

    'pos_contract_count',

    'avg_instalment',
    'max_instalment',

    'avg_instalment_future',
    'min_instalment_future',

    'avg_dpd',
    'max_dpd',

    'avg_dpd_def',
    'max_dpd_def',

    'months_balance_min',
    'months_balance_max'
]


# =========================
# FEATURE ENGINEERING
# =========================

# delay intensity
pos_agg['dpd_intensity'] = (
    pos_agg['avg_dpd'] + pos_agg['avg_dpd_def']
)

# utilization of installments
pos_agg['instalment_utilization'] = (
    pos_agg['avg_instalment_future'] / (pos_agg['avg_instalment'] + 1)
)

# activity recency
pos_agg['activity_span'] = (
    abs(pos_agg['months_balance_min'] - pos_agg['months_balance_max'])
)

# safety
pos_agg = pos_agg.replace([np.inf, -np.inf], 0).fillna(0)

In [171]:
pos_agg

,sk_id_curr,pos_contract_count,avg_instalment,max_instalment,avg_instalment_future,min_instalment_future,avg_dpd,max_dpd,avg_dpd_def,max_dpd_def,months_balance_min,months_balance_max,dpd_intensity,instalment_utilization,activity_span
0,100001,9,4.000000,4.0,1.444444,0.0,0.777778,7,0.777778,7,-96,-53,1.555556,0.288889,43
1,100002,19,24.000000,24.0,15.000000,6.0,0.000000,0,0.000000,0,-19,-1,0.000000,0.600000,18
2,100003,28,10.107143,12.0,5.785714,0.0,0.000000,0,0.000000,0,-77,-18,0.000000,0.520900,59
3,100004,4,3.750000,4.0,2.250000,0.0,0.000000,0,0.000000,0,-27,-24,0.000000,0.473684,3
4,100005,11,11.700000,12.0,7.200000,0.0,0.000000,0,0.000000,0,-25,-15,0.000000,0.566929,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
337247,456251,9,7.875000,8.0,4.375000,0.0,0.000000,0,0.000000,0,-9,-1,0.000000,0.492958,8
337248,456252,7,6.000000,6.0,3.000000,0.0,0.000000,0,0.000000,0,-82,-76,0.000000,0.428571,6
337249,456253,17,6.705882,12.0,2.000000,0.0,0.294118,5,0.294118,5,-96,-57,0.588235,0.259542,39
337250,456254,20,14.900000,16.0,10.350000,4.0,0.000000,0,0.000000,0,-11,-1,0.000000,0.650943,10


In [172]:
import sqlite3

db_path = r'C:\Users\Lenovo\Desktop\Final Project\Database\bank_credit.db'
conn = sqlite3.connect(db_path)

pos_agg.to_sql('posh_cash_final', conn, if_exists='replace', index=False)

conn.close()
print("Təbriklər! Previous Application cədvəli artıq SQL-dədir.")

Təbriklər! Previous Application cədvəli artıq SQL-dədir.


In [78]:
df_installment_payments=pd.read_csv(r"C:\Users\Lenovo\Desktop\installments_payments.csv")
df_installment_payments.head()

,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585


In [179]:
df_installment_payments['payment_delay'] = df_installment_payments['days_entry_payment'] - df_installment_payments['days_instalment']

In [180]:
df_installment_payments['payment_gap'] = df_installment_payments['amt_instalment'] - df_installment_payments['amt_payment']

In [181]:
df_installment_payments['payment_ratio'] = df_installment_payments['amt_payment'] - df_installment_payments['amt_payment']

In [182]:
df_installment_payments.columns

Index(['sk_id_prev', 'sk_id_curr', 'num_instalment_version',
       'num_instalment_number', 'days_instalment', 'days_entry_payment',
       'amt_instalment', 'amt_payment', 'pay_delay', 'pay_diff',
       'payment_delay', 'payment_gap', 'payment_ratio'],
      dtype='object')

In [183]:
inst_agg = df_installment_payments.groupby('sk_id_curr').agg({

    # activity
    'sk_id_prev': 'count',

    # delay behavior 
    'payment_delay': ['mean', 'max'],

    # financial discipline
    'payment_gap': ['mean', 'max'],

    # payment discipline
    'payment_ratio': ['mean'],

    # installment size
    'amt_instalment': ['mean'],
    'amt_payment': ['mean'],

    # timing
    'days_instalment': ['min', 'max']

}).reset_index()

In [184]:
inst_agg

sk_id_curr sk_id_prev payment_delay       payment_gap             \
                       count          mean   max        mean        max   
0          100001          7     -7.285714  11.0     0.00000      0.000   
1          100002         19    -20.421053 -12.0     0.00000      0.000   
2          100003         25     -7.160000  -1.0     0.00000      0.000   
3          100004          3     -7.666667  -3.0     0.00000      0.000   
4          100005          9    -23.555556   1.0     0.00000      0.000   
...           ...        ...           ...   ...         ...        ...   
339582     456251          7    -36.285714  -8.0     0.00000      0.000   
339583     456252          6     -2.833333   3.0     0.00000      0.000   
339584     456253         14    -14.500000   9.0   283.79250   3945.825   
339585     456254         19    -19.000000  -8.0     0.00000      0.000   
339586     456255         74     -8.067568   7.0 -6181.50223  28641.150   

       payment_ratio amt_instalment   amt_payment days_instalment          
                mean           mean          mean             min     max  
0                0.0    5885.132143   5885.132143         -2916.0 -1619.0  
1                0.0   11559.247105  11559.247105          -565.0   -25.0  
2                0.0   64754.586000  64754.586000         -2310.0  -536.0  
3                0.0    7096.155000   7096.155000          -784.0  -724.0  
4                0.0    6240.205000   6240.205000          -706.0  -466.0  
...              ...            ...           ...             ...     ...  
339582           0.0    7492.924286   7492.924286          -210.0   -30.0  
339583           0.0   10069.867500  10069.867500         -2466.0 -2316.0  
339584           0.0    4399.707857   4115.915357         -2915.0 -1716.0  
339585           0.0   10239.832895  10239.832895          -291.0    -7.0  
339586           0.0   41464.713649  47646.215878          -960.0   -66.0  

[339587 rows x 11 columns]

In [185]:
inst_agg.columns = [
    'sk_id_curr',

    'inst_count',

    'avg_payment_delay',
    'max_payment_delay',

    'avg_payment_gap',
    'max_payment_gap',

    'avg_payment_ratio',

    'avg_instalment',
    'avg_payment',

    'first_instalment_day',
    'last_instalment_day'
]

In [186]:
import sqlite3

db_path = r'C:\Users\Lenovo\Desktop\Final Project\Database\bank_credit.db'
conn = sqlite3.connect(db_path)

inst_agg.to_sql('installment_payment_final', conn, if_exists='replace', index=False)

conn.close()
print("Təbriklər! Previous Application cədvəli artıq SQL-dədir.")

Təbriklər! Previous Application cədvəli artıq SQL-dədir.


In [90]:
df_credit_card_balance=pd.read_csv(r"C:\Users\Lenovo\Desktop\credit_card_balance.csv")
df_credit_card_balance.head()

,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,AMT_PAYMENT_CURRENT,AMT_PAYMENT_TOTAL_CURRENT,AMT_RECEIVABLE_PRINCIPAL,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,1800.0,1800.0,0.000,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,2250.0,2250.0,60175.080,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,2250.0,2250.0,26926.425,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0
3,1389973,337855,-4,236572.110,225000,2250.0,2250.0,0.0,0.0,11795.760,11925.0,11925.0,224949.285,233048.970,233048.970,1.0,1,0.0,0.0,10.0,Active,0,0
4,1891521,126868,-1,453919.455,450000,0.0,11547.0,0.0,11547.0,22924.890,27000.0,27000.0,443044.395,453919.455,453919.455,0.0,1,0.0,1.0,101.0,Active,0,0


In [187]:
df_credit_card_balance.columns

Index(['sk_id_prev', 'sk_id_curr', 'months_balance', 'amt_balance',
       'amt_credit_limit_actual', 'amt_drawings_atm_current',
       'amt_drawings_current', 'amt_drawings_other_current',
       'amt_drawings_pos_current', 'amt_inst_min_regularity',
       'amt_payment_current', 'amt_payment_total_current',
       'amt_receivable_principal', 'amt_recivable', 'amt_total_receivable',
       'cnt_drawings_atm_current', 'cnt_drawings_current',
       'cnt_drawings_other_current', 'cnt_drawings_pos_current',
       'cnt_instalment_mature_cum', 'name_contract_status', 'sk_dpd',
       'sk_dpd_def', 'limit_usage_ratio'],
      dtype='object')

In [188]:
df_credit_card_balance['payment_ratio'] = df_credit_card_balance['amt_payment_total_current'] / df_credit_card_balance['amt_inst_min_regularity']
df_credit_card_balance['utilization'] =  df_credit_card_balance['amt_balance'] / df_credit_card_balance['amt_credit_limit_actual']


In [190]:
cc = df_credit_card_balance

credit_card_agg = df_credit_card_balance.groupby('sk_id_curr').agg({

    # activity
    'sk_id_prev': 'count',

    # balance behavior
    'amt_balance': ['mean', 'max'],
    'amt_credit_limit_actual': 'mean',

    # spending behavior
    'amt_drawings_current': 'sum',
    'amt_drawings_atm_current': 'sum',

    # payment behavior
    'amt_payment_current': 'mean',
    'amt_payment_total_current': 'mean',
    'amt_inst_min_regularity': 'mean',

    # risk behavior
    'sk_dpd': ['mean', 'max'],
    'sk_dpd_def': ['mean', 'max'],

    # usage
    'cnt_drawings_current': 'sum',
    'cnt_instalment_mature_cum': 'mean',

}).reset_index()

In [191]:
credit_card_agg

sk_id_curr sk_id_prev    amt_balance              \
                       count           mean         max   
0          100006          6       0.000000       0.000   
1          100011         74   54482.111149  189000.000   
2          100013         96   18159.919219  161420.220   
3          100021         17       0.000000       0.000   
4          100023          8       0.000000       0.000   
...           ...        ...            ...         ...   
103553     456244         41  131834.730732  453627.675   
103554     456246          8   13136.731875   43490.115   
103555     456247         95   23216.396211  190202.130   
103556     456248         23       0.000000       0.000   
103557     456250         12  173589.326250  200208.915   

       amt_credit_limit_actual amt_drawings_current amt_drawings_atm_current  \
                          mean                  sum                      sum   
0                270000.000000                0.000                      0.0   
1                164189.189189           180000.000                 180000.0   
2                131718.750000           571500.000                 571500.0   
3                675000.000000                0.000                      0.0   
4                135000.000000                0.000                      0.0   
...                        ...                  ...                      ...   
103553           296341.463415          1100537.910                1003500.0   
103554           135000.000000           121594.050                      0.0   
103555           144000.000000           204203.115                 202950.0   
103556           900000.000000                0.000                      0.0   
103557           178875.000000           180000.000                 180000.0   

       amt_payment_current amt_payment_total_current amt_inst_min_regularity  \
                      mean                      mean                    mean   
0                      NaN                  0.000000                0.000000   
1              4843.064189               4520.067568             3956.221849   
2              7168.346250               6817.172344             1454.539551   
3                      NaN                  0.000000                0.000000   
4                      NaN                  0.000000                0.000000   
...                    ...                       ...                     ...   
103553        32720.544878              32720.544878             6514.200000   
103554        18778.275000              15554.340000             1439.150625   
103555         4883.755263               4115.878105             1414.704789   
103556                 NaN                  0.000000                0.000000   
103557        10748.250000               1622.820000             7540.080000   

          sk_dpd     sk_dpd_def     cnt_drawings_current  \
            mean max       mean max                  sum   
0       0.000000   0   0.000000   0                    0   
1       0.000000   0   0.000000   0                    4   
2       0.010417   1   0.010417   1                   23   
3       0.000000   0   0.000000   0                    0   
4       0.000000   0   0.000000   0                    0   
...          ...  ..        ...  ..                  ...   
103553  0.000000   0   0.000000   0                   56   
103554  0.000000   0   0.000000   0                   20   
103555  0.031579   1   0.021053   1                   14   
103556  0.000000   0   0.000000   0                    0   
103557  0.000000   0   0.000000   0                    8   

       cnt_instalment_mature_cum  
                            mean  
0                       0.000000  
1                      25.767123  
2                      18.719101  
3                       0.000000  
4                       0.000000  
...                          ...  
103553                 13.600000  
103554                  3.500000  
103555                 26.

In [192]:
import sqlite3

db_path = r'C:\Users\Lenovo\Desktop\Final Project\Database\bank_credit.db'
conn = sqlite3.connect(db_path)

credit_card_agg.to_sql('credit_card_balance_final', conn, if_exists='replace', index=False)

conn.close()
print("Təbriklər! Previous Application cədvəli artıq SQL-dədir.")

Təbriklər! Previous Application cədvəli artıq SQL-dədir.


In [194]:
train=pd.read_csv(r"C:\Users\Lenovo\Desktop\application_train.csv")
train.head()

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.022,0.0198,0.0,0.0,0.0250,0.0369,0.9722,0.6243,0.0144,0.00,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.00,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.311267,0.622246,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.079,0.0554,0.0,0.0,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.01,reg oper account,block of flats,0.0714,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,1,1,1,1,1,0,Laborers,1.0,2,2,MONDAY,9,0,0,0,0,0,0,Government,NaN,0.555912,0.729567,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,

In [196]:
df = train.copy()

# =========================
# 1. LOWERCASE COLUMN NAMES
# =========================
df.columns = df.columns.str.lower()

# =========================
# 2. DROP LOW-VALUE / NOISY FEATURES
# =========================

drop_cols = [

    # mobile / constant flags
    'flag_mobil',
    'flag_emp_phone',
    'flag_work_phone',
    'flag_cont_mobile',
    'flag_phone',
    'flag_email',

    # document flags (noise)
    'flag_document_2','flag_document_3','flag_document_4','flag_document_5',
    'flag_document_6','flag_document_7','flag_document_8','flag_document_9',
    'flag_document_10','flag_document_11','flag_document_12','flag_document_13',
    'flag_document_14','flag_document_15','flag_document_16','flag_document_17',
    'flag_document_18','flag_document_19','flag_document_20','flag_document_21',

    # social circle (very noisy + missing heavy)
    'obs_30_cnt_social_circle',
    'def_30_cnt_social_circle',
    'obs_60_cnt_social_circle',
    'def_60_cnt_social_circle',

    # regional weak flags
    'reg_region_not_live_region',
    'reg_region_not_work_region',
    'live_region_not_work_region',
    'reg_city_not_live_city',
    'reg_city_not_work_city',
    'live_city_not_work_city'
]

df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

In [198]:
# ---- AGE & EMPLOYMENT ----
df['age'] = df['days_birth'] / -365

df['employment_years'] = df['days_employed']
df.loc[df['employment_years'] > 0, 'employment_years'] = np.nan  # 365243 fix
df['employment_years'] = df['employment_years'] / -365

df['days_registration_years'] = df['days_registration'] / -365


# ---- FINANCIAL RATIOS ----
df['income_to_credit'] = df['amt_income_total'] / (df['amt_credit'] + 1)

df['annuity_to_income'] = df['amt_annuity'] / (df['amt_income_total'] + 1)

df['credit_to_goods'] = df['amt_credit'] / (df['amt_goods_price'] + 1)


# ---- EXT SOURCE COMPOSITE (VERY IMPORTANT) ----
df['ext_source_mean'] = df[['ext_source_1','ext_source_2','ext_source_3']].mean(axis=1)


# ---- HOUSEHOLD ----
df['income_per_person'] = df['amt_income_total'] / (df['cnt_fam_members'] + 1)


In [200]:
df = df.replace([np.inf, -np.inf], np.nan)

# median fill for numeric
num_cols = df.select_dtypes(include=[np.number]).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())


In [201]:
df.head()

,sk_id_curr,target,name_contract_type,code_gender,flag_own_car,flag_own_realty,cnt_children,amt_income_total,amt_credit,amt_annuity,amt_goods_price,name_type_suite,name_income_type,name_education_type,name_family_status,name_housing_type,region_population_relative,days_birth,days_employed,days_registration,days_id_publish,own_car_age,occupation_type,cnt_fam_members,region_rating_client,region_rating_client_w_city,weekday_appr_process_start,hour_appr_process_start,organization_type,ext_source_1,ext_source_2,ext_source_3,apartments_avg,basementarea_avg,years_beginexpluatation_avg,years_build_avg,commonarea_avg,elevators_avg,entrances_avg,floorsmax_avg,floorsmin_avg,landarea_avg,livingapartments_avg,livingarea_avg,nonlivingapartments_avg,nonlivingarea_avg,apartments_mode,basementarea_mode,years_beginexpluatation_mode,years_build_mode,commonarea_mode,elevators_mode,entrances_mode,floorsmax_mode,floorsmin_mode,landarea_mode,livingapartments_mode,livingarea_mode,nonlivingapartments_mode,nonlivingarea_mode,apartments_medi,basementarea_medi,years_beginexpluatation_medi,years_build_medi,commonarea_medi,elevators_medi,entrances_medi,floorsmax_medi,floorsmin_medi,landarea_medi,livingapartments_medi,livingarea_medi,nonlivingapartments_medi,nonlivingarea_medi,fondkapremont_mode,housetype_mode,totalarea_mode,wallsmaterial_mode,emergencystate_mode,days_last_phone_change,amt_req_credit_bureau_hour,amt_req_credit_bureau_day,amt_req_credit_bureau_week,amt_req_credit_bureau_mon,amt_req_credit_bureau_qrt,amt_req_credit_bureau_year,age,employment_years,days_registration_years,income_to_credit,annuity_to_income,credit_to_goods,ext_source_mean,income_per_person
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,9.0,Laborers,1.0,2,2,WEDNESDAY,10,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.0220,0.0198,0.0,0.0000,0.0250,0.0369,0.9722,0.6243,0.0144,0.00,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.0000,reg oper account,block of flats,0.0149,"Stone, brick",No,-1134.0,0.0,0.0,0.0,0.0,0.0,1.0,25.920548,1.745205,9.994521,0.498034,0.121977,1.158394,0.161787,101250.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,9.0,Core staff,2.0,1,1,MONDAY,11,School,0.311267,0.622246,0.535276,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.0790,0.0554,0.0,0.0000,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.0100,reg oper account,block of flats,0.0714,Block,No,-828.0,0.0,0.0,0.0,0.0,0.0,0.0,45.931507,3.254795,3.249315,0.208735,0.132216,1.145198,0.466757,90000.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,135000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.010032,-19046,-225,-4260.0,-2531,26.0,Laborers,1.0,2,2,MONDAY,9,Government,0.505998,0.555912,0.729567,0.0876,0.0763,0.9816,0.7552,0.0211,0.00,0.1379,0.1667,0.2083,0.0481,0.0756,0.0745,0.0000,0.0036,0.0840,0.0746,0.9816,0.7648,0.0190,0.0000,0.1379,0.1667,0.2083,0.0458,0.0771,0.0731,0.0,0.0011,0.0864,0.0758,0.9816,0.7585,0.0208,0.00,0.1379,0.1667,0.2083,0.0487,0.0761,0.0749,0.0000,0.0031,NaN,NaN,0.0688,NaN,NaN,-815.0,0.0,0.0,0.0,0.0,0.0,0.0,52.180822,0.616438,11.671233,0.499996,0.099999,0.999993,0.642739,33750.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,297000.0,Unaccompanied,Working,Secondary / secondary special,Civil marriage,House / apartment,0.008019,-19005,-3039,-9833.0,-2437,9.0,Laborers,2.0,2,2,WEDNESDAY,17,Business Entity Type 3,0.505998,0.650442,0

array(['Laborers', 'Core staff', 'Accountants', 'Managers', nan,
       'Drivers', 'Sales staff', 'Cleaning staff', 'Cooking staff',
       'Private service staff', 'Medicine staff', 'Security staff',
       'High skill tech staff', 'Waiters/barmen staff',
       'Low-skill Laborers', 'Realty agents', 'Secretaries', 'IT staff',
       'HR staff'], dtype=object)

In [205]:
import sqlite3

db_path = r'C:\Users\Lenovo\Desktop\Final Project\Database\bank_credit.db'
conn = sqlite3.connect(db_path)

df.to_sql('train_application', conn, if_exists='replace', index=False)

conn.close()
print("Təbriklər! Previous Application cədvəli artıq SQL-dədir.")

Təbriklər! Previous Application cədvəli artıq SQL-dədir.


In [207]:
def flatten_columns(df):
    df.columns = [
        '_'.join(col).strip() if isinstance(col, tuple) else col
        for col in df.columns
    ]
    return df

In [208]:
bureau_agg = flatten_columns(bureau_agg)
prev_agg = flatten_columns(prev_agg)
pos_agg = flatten_columns(pos_agg)
inst_agg = flatten_columns(inst_agg)
credit_card_agg = flatten_columns(credit_card_agg)

In [210]:
for df_tmp in [bureau_agg, prev_agg, pos_agg, inst_agg, credit_card_agg]:
    df_tmp.columns = df_tmp.columns.str.lower()

In [214]:
inst_agg = inst_agg.reset_index()
inst_agg.columns = inst_agg.columns.str.lower()

In [215]:
inst_agg = inst_agg.reset_index()
inst_agg.columns = inst_agg.columns.str.lower()

In [216]:
def prefix_df(df, prefix):
    df = df.reset_index()
    df.columns = df.columns.str.lower()

    for col in df.columns:
        if col != "sk_id_curr":
            df = df.rename(columns={col: f"{prefix}_{col}"})
    return df

In [218]:
def prefix_df(df, prefix):

    df = df.copy()

    # ONLY reset if index is not default
    if df.index.name is not None or isinstance(df.index, pd.MultiIndex):
        df = df.reset_index()

    # remove old weird columns if exist
    df = df.loc[:, ~df.columns.duplicated()]

    df.columns = df.columns.str.lower()

    # rename all except key
    for col in df.columns:
        if col != "sk_id_curr":
            df = df.rename(columns={col: f"{prefix}_{col}"})

    return df

In [219]:
inst_agg = prefix_df(inst_agg, "inst")
pos_agg = prefix_df(pos_agg, "pos")
bureau_agg = prefix_df(bureau_agg, "bur")
prev_agg = prefix_df(prev_agg, "prev")
credit_card_agg = prefix_df(credit_card_agg, "cc")

In [222]:
def fix_keys(df):
    df = df.copy()

    # index → column
    df = df.reset_index()

    # lowercase
    df.columns = df.columns.str.lower()

    # find key if missing
    if 'sk_id_curr' not in df.columns:
        for c in df.columns:
            if 'sk_id' in c and 'curr' in c:
                df = df.rename(columns={c: 'sk_id_curr'})
                break

    return df

In [223]:
credit_card_agg = fix_keys(credit_card_agg)

In [224]:
df = df.merge(bureau_agg, on='sk_id_curr', how='left')

df = df.merge(prev_agg, on='sk_id_curr', how='left')

df = df.merge(pos_agg, on='sk_id_curr', how='left')

df = df.merge(inst_agg, on='sk_id_curr', how='left')

df = df.merge(credit_card_agg, on='sk_id_curr', how='left')

In [233]:
df.to_csv(r"C:\Users\Lenovo\Desktop\bank_credit_dataset.csv",index=False)